In [1]:
import anndata as ad
import pandas as pd
import numpy as np
from scipy.io import mmread, mmwrite
from scipy.sparse import csr_matrix
import muon
import scarches as sca
from multigrate.data import organize_multiome_anndatas
import scanpy as sc
import scib_metrics
from typing import Optional
import os, sys
from scipy.sparse import csr_matrix, coo_matrix
import scipy
from scipy import sparse
import importlib
import matplotlib.pyplot as plt
import seaborn as sns
import scib
import scib_metrics
from scib_metrics.benchmark import Benchmarker
from typing import Any, Callable, Optional, Union
from plottable import ColumnDefinition, Table
from plottable.cmap import normed_cmap
from plottable.plots import bar

import warnings
warnings.filterwarnings("ignore")

/home/zhouweige/anaconda3/envs/scib/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 captum (see https://github.com/pytorch/captum).


In [2]:
def read_RNA_ATAC(RNA_path,ATAC_path):
    # gene expression
    cell_names = pd.read_csv(RNA_path+'/barcodes.tsv', sep = '\t', header=None, index_col=None)
    cell_names.columns =  ['cell_ids'] 
    cell_names['cell_ids'] = cell_names['cell_ids'].str.replace('.','-')
    X = csr_matrix(mmread(RNA_path+'/matrix.mtx').T)
    gene_names = pd.read_csv(RNA_path+'/features.tsv', sep = '\t',  header=None, index_col=None) 
    gene_names.columns =  ['gene_ids'] 
    adata_RNA = ad.AnnData(X, obs=pd.DataFrame(index=cell_names.cell_ids), var=pd.DataFrame(index = gene_names.gene_ids))
    adata_RNA.var_names_make_unique()
    # peak information
    cell_names = pd.read_csv(ATAC_path + '/barcodes.tsv', sep = '\t', header=None, index_col=None)
    cell_names.columns =  ['cell_ids'] 
    cell_names['cell_ids'] = cell_names['cell_ids'].str.replace('.','-')
    X = csr_matrix(mmread(ATAC_path + '/matrix.mtx').T)
    peak_name = pd.read_csv(ATAC_path + '/features.tsv', sep = '\t',header=None,index_col=None)
    peak_name.columns = ['peak_ids']
    adata_ATAC  = ad.AnnData(X, obs=pd.DataFrame(index=cell_names.cell_ids), var=pd.DataFrame(index = peak_name.peak_ids))
    return adata_RNA, adata_ATAC

In [3]:
import os
import numpy as np
import scanpy as sc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def pca(adata, use_reps=None, n_comps=10):

    """Dimension reduction with PCA algorithm"""

    from sklearn.decomposition import PCA
    from scipy.sparse.csc import csc_matrix
    from scipy.sparse.csr import csr_matrix
    pca = PCA(n_components=n_comps)
    if use_reps is not None:
       feat_pca = pca.fit_transform(adata.obsm[use_reps])
    else:
       if isinstance(adata.X, csc_matrix) or isinstance(adata.X, csr_matrix):
          feat_pca = pca.fit_transform(adata.X.toarray())
       else:
          feat_pca = pca.fit_transform(adata.X)

    return feat_pca

def supervised_clustering(adata, n_clusters=7, key='Garfield', add_key='Garfield_cluster', cluster_method='leiden',
               start=0.5, end=2.0, increment=0.05, use_pca=False, n_comps=20):
    if use_pca:
       adata.obsm[key + '_pca'] = pca(adata, use_reps=key, n_comps=n_comps)

    method_cluster = add_key
    method = key
    
    if cluster_method == 'leiden':
       if use_pca:
          res = search_res(adata, n_clusters, use_rep=key + '_pca', cluster_method=cluster_method, 
                           start=start, end=end, increment=increment)
       else:
          res = search_res(adata, n_clusters, use_rep=method, cluster_method=cluster_method, 
                           start=start, end=end, increment=increment)
       sc.tl.leiden(adata, random_state=0, key_added=method_cluster, resolution=res, neighbors_key=method)
    elif cluster_method == 'louvain':
       if use_pca:
          res = search_res(adata, n_clusters, use_rep=key + '_pca', cluster_method=cluster_method, 
                           start=start, end=end, increment=increment)
       else:
          res = search_res(adata, n_clusters, use_rep=method, cluster_method=cluster_method, 
                           start=start, end=end, increment=increment)
       sc.tl.louvain(adata, random_state=0, key_added=method_cluster, resolution=res, neighbors_key=method)

def search_res(adata, n_clusters, use_rep='Garfield', cluster_method='leiden', start=0.5, end=2.0,
               increment=0.05, expansion_factor=1.75, max_iter=50):
    '''\
    Searching corresponding resolution according to given cluster number

    Parameters
    ----------
    adata : anndata
        AnnData object of spatial data.
    n_clusters : int
        Target number of clusters.
    method : string
        Tool for clustering. Supported tools include 'leiden' and 'louvain'. The default is 'leiden'.
    use_rep : string
        The indicated representation for clustering.
    start : float
        The start value for searching.
    end : float
        The end value for searching.
    increment : float
        The step size to increase.
    expansion_factor : float
        Factor to expand the search range if no resolution is found in the initial range.
    max_iter : int
        Maximum number of iterations to prevent infinite loops.

    Returns
    -------
    res : float
        Resolution.

    '''
    print('Searching resolution...')
    label = 0
    method = use_rep
    method_cluster = method + '_cluster'
    
    current_start, current_end = start, end
    best_res = None
    best_diff = float('inf')  # Track the best resolution so far
    best_count = 0
    iteration = 0  # Track number of iterations
    
    # 构图
    sc.pp.neighbors(adata, use_rep=use_rep, key_added=method)

    while label == 0 and iteration < max_iter:
        iteration += 1
        for res in sorted(list(np.arange(current_start, current_end, increment)), reverse=True):
            if cluster_method == 'leiden':
                sc.tl.leiden(adata, random_state=0, key_added=method_cluster,
                             resolution=res, neighbors_key=method)
                count_unique = adata.obs[method_cluster].nunique()
            elif cluster_method == 'louvain':
                sc.tl.louvain(adata,  random_state=0, key_added=method_cluster, 
                             resolution=method_reso, neighbors_key=method)
                count_unique = adata.obs[method_cluster].nunique()

            print(f'resolution={res}, cluster number={count_unique}')

            # Update the best resolution if it's closer to the target
            diff = abs(count_unique - n_clusters)
            if diff < best_diff:
                best_diff = diff
                best_res = res
                best_count = count_unique

            if count_unique == n_clusters:
                label = 1
                break

        if label == 0:
            if best_res is not None:
                # Instead of starting from scratch, continue from the closest found resolution
                if best_count > n_clusters:
                    current_start = min(best_res, current_start * 0.9)  # Avoid too small a search range
                    current_end = min(current_start * expansion_factor, end * 2)  # Limit max range
                    print(f"Expanding search range: new range ({current_start}, {current_end})")
                else:
                    current_start = max(best_res, current_start * 1.2)  # Avoid too small a search range
                    current_end = min(current_start * expansion_factor, end * 2)  # Limit max range
                    print(f"Expanding search range: new range ({current_start}, {current_end})")
            else:
                print("Warning: No valid resolution found yet, expanding the search range.")
                current_start *= expansion_factor
                current_end *= expansion_factor

    if iteration >= max_iter:
        print("Warning: Reached maximum iteration limit.")

    return best_res if label == 1 else None

In [4]:
def count_metrics_for_resolution(dataset, reso_dict=None, start=0.5, end=2.0, increment=0.05):
    rna_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/datasets/MultiomicsBenchmark/Raw_data/RNA_ATAC/{dataset}/RNA'
    atac_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/datasets/MultiomicsBenchmark/Raw_data/RNA_ATAC/{dataset}/ATAC'
    metadata_file = f'/data2/zhouwg_data/project/Garfield_benchmark/datasets/MultiomicsBenchmark/Raw_data/RNA_ATAC/{dataset}/meta_data.csv'
    res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC/{dataset}'

    # 读取 RNA 和 ATAC 数据
    adata_RNA, adata_ATAC = read_RNA_ATAC(rna_dir, atac_dir)

    # 处理 RNA 数据
    adata_RNA.layers["counts"] = adata_RNA.X.copy()
    sc.pp.normalize_total(adata_RNA)
    sc.pp.log1p(adata_RNA)
    sc.pp.highly_variable_genes(adata_RNA, flavor="seurat_v3", n_top_genes=3000, subset=False)
    adata_RNA = adata_RNA[:, adata_RNA.var.highly_variable].copy()

    # 处理 ATAC 数据
    adata_ATAC.layers['counts'] = adata_ATAC.X.copy()
    sc.pp.normalize_total(adata_ATAC, target_sum=1e4)
    sc.pp.log1p(adata_ATAC)
    adata_ATAC.layers['log-norm'] = adata_ATAC.X.copy()
    sc.pp.highly_variable_genes(adata_ATAC, n_top_genes=10000)
    adata_ATAC = adata_ATAC[:, adata_ATAC.var.highly_variable].copy()

    # 合并 RNA 和 ATAC 数据
    adata = organize_multiome_anndatas(
        adatas=[[adata_RNA], [adata_ATAC]],    # RNA-seq 始终在第一位
        layers=[['counts'], ['log-norm']]      # 使用 .layers 中的数据
    )

    # 读取元数据
    metadata = pd.read_csv(metadata_file, index_col=0)
    metadata.index = adata.obs_names  # 修正元数据的索引
    adata.obs['cell_type'] = metadata['celltype'].astype('category')

    # 处理缺失的 cell_type
    if adata.obs["cell_type"].isna().sum() != 0:
        adata.obs["cell_type"] = adata.obs["cell_type"].cat.add_categories(['NaN'])
        adata.obs.loc[adata.obs["cell_type"].isna(), "cell_type"] = 'NaN'

    adata.obs['batch'] = ['batch1'] * adata.shape[0]

    result = pd.DataFrame()
    metrics_list = []
    method_list = ['Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat']

    # 计算不同方法的指标
    for method in method_list:
        method_file = f'{res_dir}/{method}.csv'
        if os.path.exists(method_file):
            latent = pd.read_csv(method_file, header=None)
            latent.index = adata.obs_names
            adata.obsm[method] = latent
            print('the length of cell type:', adata.obs['cell_type'].nunique())
            method_cluster = method+'_cluster'
            if reso_dict is not None and dataset in reso_dict.keys():
                data_reso_dict = reso_dict[dataset]
                method_reso = data_reso_dict[method]
                sc.pp.neighbors(adata, use_rep=method, key_added=method)
                if method_reso is not None:
                    print(f'method_reso is not None, using the {method_reso} as resolution for {method}...')
                    sc.tl.leiden(adata,  random_state=0, key_added=method_cluster, 
                                 resolution=method_reso, neighbors_key=method)
                    sc.tl.umap(adata, neighbors_key=method)
                    print('Number of clusters:', adata.obs[method_cluster].nunique())
                else:
                    print('Searching resolution for {}...'.format(method))
                    supervised_clustering(adata, n_clusters=adata.obs['cell_type'].nunique(),
                                          key=method, add_key=method_cluster, cluster_method='leiden',
                                          start=start, end=end, increment=increment)
                    sc.tl.umap(adata, neighbors_key=method)
                    print('Number of clusters:', adata.obs[method_cluster].nunique())
            else:
                print('Searching resolution for {}...'.format(method))
                supervised_clustering(adata, n_clusters=adata.obs['cell_type'].nunique(),
                                      key=method, add_key=method_cluster, cluster_method='leiden',
                                      start=start, end=end, increment=increment)
                sc.tl.umap(adata, neighbors_key=method)
                print('Number of clusters:', adata.obs[method_cluster].nunique())
            # 计算指标
            # scib.metrics.cluster_optimal_resolution(adata, cluster_key="cluster", label_key="cell_type")
            ari = scib.metrics.ari(adata, cluster_key=method_cluster, label_key="cell_type")
            iso_asw = scib.metrics.isolated_labels_asw(adata, label_key="cell_type", batch_key='batch', embed=method, verbose=False)
            nmi = scib.metrics.nmi(adata, cluster_key=method_cluster, label_key="cell_type")
            clisi = scib.metrics.clisi_graph(adata, label_key="cell_type", use_rep=method, type_='embed')
            sht = scib.metrics.silhouette(adata, label_key="cell_type", embed=method, scale=True) #  metric='euclidean',

            metrics_list.append([ari, iso_asw, nmi, clisi, sht, method])

    # 处理 Seurat 的结果 通过 graph 的方式
    if method == 'Seurat_graph':
        method = 'Seurat'
        con = mmread(f'{res_dir}/{method}_connectivities.mtx')
        dis = mmread(f'{res_dir}/{method}_distance.mtx')
        
        # uns['neighbors'] 的信息
        adata.uns['neighbors'] = {
            'connectivities_key': 'connectivities', 'distances_key': 'distances',
            'params': {'n_neighbors': 20, 'method': 'umap', 'random_state': 0, 'metric': 'euclidean'}
        }
        adata.uns['neighbors']['distances'] = csr_matrix(dis)
        adata.uns['neighbors']['connectivities'] = csr_matrix(con)
        adata.obsp['distances'] = csr_matrix(dis)
        adata.obsp['connectivities'] = csr_matrix(con)
        # Seurat 的信息   
        adata.uns['Seurat'] = {
            'connectivities_key': 'Seurat_connectivities', 'distances_key': 'Seurat_distances',
            'params': {'n_neighbors': 20, 'method': 'umap', 'random_state': 0, 'metric': 'euclidean', 'use_rep': 'Seurat'}
        }
        adata.uns['Seurat']['Seurat_distances'] = csr_matrix(dis)
        adata.uns['Seurat']['Seurat_connectivities'] = csr_matrix(con)
        adata.obsp['Seurat_distances'] = csr_matrix(dis)
        adata.obsp['Seurat_connectivities'] = csr_matrix(con)
        
        sc.tl.umap(adata) # , neighbors_key=method
        adata.obsm['Seurat'] = adata.obsm['X_umap'][:, :2]  # 确保只取二维的 UMAP 结果
        method_cluster = method+'_cluster'
        if reso_dict is not None and dataset in reso_dict.keys():
            data_reso_dict = reso_dict[dataset]
            method_reso = data_reso_dict[method]
            if method_reso is not None:
                print(f'method_reso is not None, using the {method_reso} as resolution for {method}...')
                sc.tl.leiden(adata,  random_state=0, key_added=method_cluster, 
                             resolution=method_reso, neighbors_key=method)
                print('Number of clusters:', adata.obs[method_cluster].nunique())
            else:
                print('Searching resolution for Seurat...')
                start = 0.3
                end = 1.5
                increment = 0.05
                for res in sorted(list(np.arange(start, end, increment)), reverse=True):
                   sc.tl.leiden(adata, random_state=0, resolution=res, neighbors_key='Seurat')
                   count_unique = adata.obs['leiden'].nunique()
                   print('resolution={}, cluster number={}'.format(res, count_unique))
                   if count_unique == adata.obs['cell_type'].nunique():
                        break
                print('resolution for Seurat:', res)
                sc.tl.leiden(adata, random_state=0, resolution=res, neighbors_key='Seurat')
                print('Number of clusters:', adata.obs['leiden'].nunique())
                adata.obs[method_cluster] = adata.obs['leiden']
        else:
            print('Searching resolution for Seurat...')
            start = 0.3
            end = 1.5
            increment = 0.05
            for res in sorted(list(np.arange(start, end, increment)), reverse=True):
               sc.tl.leiden(adata, random_state=0, resolution=res, neighbors_key='Seurat')
               count_unique = adata.obs['leiden'].nunique()
               print('resolution={}, cluster number={}'.format(res, count_unique))
               if count_unique == adata.obs['cell_type'].nunique():
                    break
            print('resolution for Seurat:', res)
            sc.tl.leiden(adata, random_state=0, resolution=res, neighbors_key='Seurat')
            print('Number of clusters:', adata.obs['leiden'].nunique())
            adata.obs[method_cluster] = adata.obs['leiden']
    
        # 计算 Seurat 的指标
        # scib.metrics.cluster_optimal_resolution(adata, cluster_key="cluster", label_key="cell_type")
        ari = scib.metrics.ari(adata, cluster_key=method_cluster, label_key="cell_type")
        iso_asw = scib.metrics.isolated_labels_asw(adata, label_key="cell_type", batch_key='batch', embed=method, verbose=False)
        nmi = scib.metrics.nmi(adata, cluster_key=method_cluster, label_key="cell_type")
        clisi = scib.metrics.clisi_graph(adata, label_key="cell_type", use_rep=method, type_='embed')
        sht = scib.metrics.silhouette(adata, label_key="cell_type", embed=method, scale=True) # metric='euclidean',
        metrics_list.append([ari, iso_asw, nmi, clisi, sht, 'Seurat'])

    # 保存结果
    df = pd.DataFrame(metrics_list, columns=['ARI', 'Isolated_Labels_ASW', 'NMI', 'cLISI_Graph', 'Silhouette', 'method'])
    result = pd.concat([result, df], ignore_index=True)
    result['Dataset'] = dataset
    result.to_csv(f'{res_dir}/../{dataset}_metrics_result.csv', index=False)
    print(dataset)

    return adata

In [10]:
def count_metrics_for_best_resolution(dataset):
    rna_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/datasets/MultiomicsBenchmark/Raw_data/RNA_ATAC/{dataset}/RNA'
    atac_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/datasets/MultiomicsBenchmark/Raw_data/RNA_ATAC/{dataset}/ATAC'
    metadata_file = f'/data2/zhouwg_data/project/Garfield_benchmark/datasets/MultiomicsBenchmark/Raw_data/RNA_ATAC/{dataset}/meta_data.csv'
    res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC/{dataset}'

    # 读取 RNA 和 ATAC 数据
    adata_RNA, adata_ATAC = read_RNA_ATAC(rna_dir, atac_dir)

    # 处理 RNA 数据
    adata_RNA.layers["counts"] = adata_RNA.X.copy()
    sc.pp.normalize_total(adata_RNA)
    sc.pp.log1p(adata_RNA)
    sc.pp.highly_variable_genes(adata_RNA, flavor="seurat_v3", n_top_genes=3000, subset=False)
    adata_RNA = adata_RNA[:, adata_RNA.var.highly_variable].copy()

    # 处理 ATAC 数据
    adata_ATAC.layers['counts'] = adata_ATAC.X.copy()
    sc.pp.normalize_total(adata_ATAC, target_sum=1e4)
    sc.pp.log1p(adata_ATAC)
    adata_ATAC.layers['log-norm'] = adata_ATAC.X.copy()
    sc.pp.highly_variable_genes(adata_ATAC, n_top_genes=10000)
    adata_ATAC = adata_ATAC[:, adata_ATAC.var.highly_variable].copy()

    # 合并 RNA 和 ATAC 数据
    adata = organize_multiome_anndatas(
        adatas=[[adata_RNA], [adata_ATAC]],    # RNA-seq 始终在第一位
        layers=[['counts'], ['log-norm']]      # 使用 .layers 中的数据
    )

    # 读取元数据
    metadata = pd.read_csv(metadata_file, index_col=0)
    metadata.index = adata.obs_names  # 修正元数据的索引
    adata.obs['cell_type'] = metadata['celltype'].astype('category')

    # 处理缺失的 cell_type
    if adata.obs["cell_type"].isna().sum() != 0:
        adata.obs["cell_type"] = adata.obs["cell_type"].cat.add_categories(['NaN'])
        adata.obs.loc[adata.obs["cell_type"].isna(), "cell_type"] = 'NaN'

    adata.obs['batch'] = ['batch1'] * adata.shape[0]

    result = pd.DataFrame()
    metrics_list = []
    method_list = ['Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat']
    for method in method_list:
        method_file = f'{res_dir}/{method}.csv'
        if os.path.exists(method_file):
            latent = pd.read_csv(method_file, header=None)
            latent.index = adata.obs_names
            adata.obsm[method] = latent
            sc.pp.neighbors(adata, use_rep=method)
            sc.tl.umap(adata)
            # sc.tl.leiden(adata, key_added="cluster")
            scib.metrics.cluster_optimal_resolution(adata, cluster_key="cluster", label_key="cell_type")
            ari = scib.metrics.ari(adata, cluster_key="cluster", label_key="cell_type")
            iso_asw = scib.metrics.isolated_labels_asw(adata, label_key="cell_type", batch_key='batch', embed=method,  verbose = False)
            nmi = scib.metrics.nmi(adata, cluster_key="cluster", label_key="cell_type")
            clisi = scib.metrics.clisi_graph(adata, label_key="cell_type",use_rep=method, type_='embed')
            sht = scib.metrics.silhouette(adata, label_key="cell_type", embed=method, metric='euclidean', scale=True)
            metrics_list.append([ari, iso_asw, nmi, clisi, sht, method])

    # 保存结果
    df = pd.DataFrame(metrics_list, columns=['ARI', 'Isolated_Labels_ASW', 'NMI', 'cLISI_Graph', 'Silhouette', 'method'])
    result = pd.concat([result, df], ignore_index=True)
    result['Dataset'] = dataset
    result.to_csv(f'{res_dir}/../{dataset}_metrics_result_stable_reso.csv', index=False)
    print(dataset)
    
    return adata

### Not 数据集 10_GSE201402_down

In [31]:
## 记录resolution
reso_dict = {
    '10_GSE201402_down': {
        'Garfield': 0.55,
        'Multigrate': 0.8500000000000003,
        'MultiVI': 0.2657205000000001,
        'MOFA': 0.45,
        'Seurat': 0.01390641947218469, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('10_GSE201402_down', reso_dict=reso_dict)
adata

the length of cell type: 16
Searching resolution for Garfield...
Searching resolution...
resolution=1.9500000000000013, cluster number=33
resolution=1.9000000000000012, cluster number=34
resolution=1.8500000000000012, cluster number=33
resolution=1.8000000000000012, cluster number=33
resolution=1.750000000000001, cluster number=30
resolution=1.700000000000001, cluster number=31
resolution=1.650000000000001, cluster number=29
resolution=1.600000000000001, cluster number=29
resolution=1.550000000000001, cluster number=28
resolution=1.5000000000000009, cluster number=26
resolution=1.4500000000000008, cluster number=26
resolution=1.4000000000000008, cluster number=26
resolution=1.3500000000000008, cluster number=25
resolution=1.3000000000000007, cluster number=24
resolution=1.2500000000000007, cluster number=24
resolution=1.2000000000000006, cluster number=24
resolution=1.1500000000000006, cluster number=24
resolution=1.1000000000000005, cluster number=23
resolution=1.0500000000000005, clu

AnnData object with n_obs × n_vars = 9383 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [32]:
import matplotlib.pyplot as plt

dataset = '10_GSE201402_down'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [33]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '10_GSE201402_down'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.140319,0.588752,0.524695,1.0,0.734643,Garfield,10_GSE201402_down
1,0.248909,0.591281,0.622897,1.0,0.627959,Multigrate,10_GSE201402_down
2,0.197756,0.761025,0.583809,1.0,0.785782,MultiVI,10_GSE201402_down
3,0.264119,0.676466,0.633253,1.0,0.788906,MOFA,10_GSE201402_down
4,0.477558,0.831389,0.745378,1.0,0.761591,Seurat,10_GSE201402_down


#### 探索一下用最佳分辨率

In [20]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('10_GSE201402_down')
adata

resolution: 0.1, nmi: 0.707773672061088
resolution: 0.2, nmi: 0.7188311447696103
resolution: 0.3, nmi: 0.7188311447696103
resolution: 0.4, nmi: 0.7199456855325836
resolution: 0.5, nmi: 0.7249505132333783
resolution: 0.6, nmi: 0.7078065102280388
resolution: 0.7, nmi: 0.7149290383827273
resolution: 0.8, nmi: 0.7022986337209329
resolution: 0.9, nmi: 0.6711186766525109
resolution: 1.0, nmi: 0.6790145617863353
resolution: 1.1, nmi: 0.6656990046238804
resolution: 1.2, nmi: 0.6696040322364775
resolution: 1.3, nmi: 0.6644144153347636
resolution: 1.4, nmi: 0.6534915586972398
resolution: 1.5, nmi: 0.6502094566335053
resolution: 1.6, nmi: 0.6579376494847752
resolution: 1.7, nmi: 0.6439820276067458
resolution: 1.8, nmi: 0.6122456835491061
resolution: 1.9, nmi: 0.6262365923325991
resolution: 2.0, nmi: 0.6058718893678586
optimised clustering against cell_type
optimal cluster resolution: 0.5
optimal score: 0.7249505132333783
resolution: 0.1, nmi: 0.6158810825337998
resolution: 0.2, nmi: 0.67908029491

AnnData object with n_obs × n_vars = 2413 × 13000
    obs: 'group', 'cell_type', 'batch', 'silhouette_temp', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [ ]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '10_GSE201402_down'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

### 数据集 9_pbmc_unsorted_3k

In [26]:
## 记录resolution
reso_dict = {
    '9_pbmc_unsorted_3k': {
        'Garfield': 1.2500000000000007,
        'Multigrate': 1.700000000000001,
        'MultiVI': 0.455,
        'MOFA': 0.45,
        'Seurat': 0.14121476824050008, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('9_pbmc_unsorted_3k', reso_dict=reso_dict)
adata

the length of cell type: 12
Searching resolution for Garfield...
Searching resolution...
resolution=1.9500000000000013, cluster number=17
resolution=1.9000000000000012, cluster number=16
resolution=1.8500000000000012, cluster number=16
resolution=1.8000000000000012, cluster number=16
resolution=1.750000000000001, cluster number=16
resolution=1.700000000000001, cluster number=16
resolution=1.650000000000001, cluster number=14
resolution=1.600000000000001, cluster number=15
resolution=1.550000000000001, cluster number=15
resolution=1.5000000000000009, cluster number=15
resolution=1.4500000000000008, cluster number=14
resolution=1.4000000000000008, cluster number=15
resolution=1.3500000000000008, cluster number=14
resolution=1.3000000000000007, cluster number=13
resolution=1.2500000000000007, cluster number=12
Number of clusters: 12
the length of cell type: 12
method_reso is not None, using the 1.700000000000001 as resolution for Multigrate...
Number of clusters: 12
the length of cell typ

AnnData object with n_obs × n_vars = 2413 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [27]:
import matplotlib.pyplot as plt

dataset = '9_pbmc_unsorted_3k'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [28]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '9_pbmc_unsorted_3k'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.455839,0.559999,0.664593,0.958354,0.507223,Garfield,9_pbmc_unsorted_3k
1,0.270386,0.505980,0.513119,0.879847,0.503597,Multigrate,9_pbmc_unsorted_3k
2,0.486626,0.707436,0.729399,0.993246,0.658809,MultiVI,9_pbmc_unsorted_3k
3,0.519998,0.608050,0.674235,0.981996,0.575843,MOFA,9_pbmc_unsorted_3k
4,0.626682,0.795544,0.779000,0.997477,0.702561,Seurat,9_pbmc_unsorted_3k


#### 使用默认固定的resolution

In [11]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('9_pbmc_unsorted_3k')
adata

Chunk 138 does not have enough neighbors. Skipping...
Chunk 142 does not have enough neighbors. Skipping...
Chunk 204 does not have enough neighbors. Skipping...
Chunk 352 does not have enough neighbors. Skipping...
Chunk 379 does not have enough neighbors. Skipping...
Chunk 421 does not have enough neighbors. Skipping...
Chunk 804 does not have enough neighbors. Skipping...
Chunk 1016 does not have enough neighbors. Skipping...
Chunk 1491 does not have enough neighbors. Skipping...
Chunk 1754 does not have enough neighbors. Skipping...
Chunk 1889 does not have enough neighbors. Skipping...
Chunk 1928 does not have enough neighbors. Skipping...
Chunk 1990 does not have enough neighbors. Skipping...
Chunk 2070 does not have enough neighbors. Skipping...
Chunk 2094 does not have enough neighbors. Skipping...
Chunk 2146 does not have enough neighbors. Skipping...
Chunk 2259 does not have enough neighbors. Skipping...
Chunk 2383 does not have enough neighbors. Skipping...
Chunk 2413 does n

AnnData object with n_obs × n_vars = 2413 × 13000
    obs: 'group', 'cell_type', 'batch', 'cluster', 'silhouette_temp'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [12]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '9_pbmc_unsorted_3k'

# df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_stable_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.539063,0.559999,0.718015,0.958354,0.507223,Garfield,9_pbmc_unsorted_3k
1,0.396722,0.505980,0.586223,0.879847,0.503597,Multigrate,9_pbmc_unsorted_3k
2,0.318746,0.707436,0.670271,0.993246,0.658809,MultiVI,9_pbmc_unsorted_3k
3,0.394702,0.608050,0.662568,0.981996,0.575843,MOFA,9_pbmc_unsorted_3k
4,0.301517,0.795544,0.688714,0.997477,0.702561,Seurat,9_pbmc_unsorted_3k


#### 利用最佳分辨率计算指标

In [6]:
## 记录resolution
reso_dict = {
    '9_pbmc_unsorted_3k': {
        'Garfield': 0.5, # 1.2500000000000007,
        'Multigrate': 1.700000000000001,
        'MultiVI': 0.455,
        'MOFA': 0.45,
        'Seurat': 0.14121476824050008, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('9_pbmc_unsorted_3k', reso_dict=reso_dict)
adata

the length of cell type: 12
method_reso is not None, using the 0.5 as resolution for Garfield...
Number of clusters: 7
the length of cell type: 12
method_reso is not None, using the 1.700000000000001 as resolution for Multigrate...
Number of clusters: 12
the length of cell type: 12
method_reso is not None, using the 0.455 as resolution for MultiVI...
Number of clusters: 12
the length of cell type: 12
method_reso is not None, using the 0.45 as resolution for MOFA...
Number of clusters: 12
the length of cell type: 12
method_reso is not None, using the 0.14121476824050008 as resolution for Seurat...
Number of clusters: 12
Chunk 138 does not have enough neighbors. Skipping...
Chunk 142 does not have enough neighbors. Skipping...
Chunk 204 does not have enough neighbors. Skipping...
Chunk 352 does not have enough neighbors. Skipping...
Chunk 379 does not have enough neighbors. Skipping...
Chunk 421 does not have enough neighbors. Skipping...
Chunk 804 does not have enough neighbors. Skippin

AnnData object with n_obs × n_vars = 2413 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [7]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '9_pbmc_unsorted_3k'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.698596,0.559999,0.755856,0.958354,0.507223,Garfield,9_pbmc_unsorted_3k
1,0.270386,0.505980,0.513119,0.879847,0.503597,Multigrate,9_pbmc_unsorted_3k
2,0.486626,0.707436,0.729399,0.993246,0.658809,MultiVI,9_pbmc_unsorted_3k
3,0.519998,0.608050,0.674235,0.981996,0.575843,MOFA,9_pbmc_unsorted_3k
4,0.626682,0.795544,0.779000,0.997477,0.702561,Seurat,9_pbmc_unsorted_3k


### 数据集 8_pbmc_unsorted_10k

In [12]:
## 记录resolution
reso_dict = {
    '8_pbmc_unsorted_10k': {
        'Garfield': 0.66,
        'Multigrate': 1.750000000000001,
        'MultiVI': 0.45,
        'MOFA': 0.8000000000000003,
        'Seurat': 0.14121476824050008, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('8_pbmc_unsorted_10k', reso_dict=reso_dict)
adata

the length of cell type: 17
method_reso is not None, using the 0.66 as resolution for Garfield...
Number of clusters: 17
the length of cell type: 17
Searching resolution for Multigrate...
Searching resolution...
resolution=1.9500000000000013, cluster number=18
resolution=1.9000000000000012, cluster number=18
resolution=1.8500000000000012, cluster number=18
resolution=1.8000000000000012, cluster number=18
resolution=1.750000000000001, cluster number=17
Number of clusters: 17
the length of cell type: 17
Searching resolution for MultiVI...
Searching resolution...
resolution=1.9500000000000013, cluster number=44
resolution=1.9000000000000012, cluster number=42
resolution=1.8500000000000012, cluster number=42
resolution=1.8000000000000012, cluster number=40
resolution=1.750000000000001, cluster number=40
resolution=1.700000000000001, cluster number=42
resolution=1.650000000000001, cluster number=41
resolution=1.600000000000001, cluster number=41
resolution=1.550000000000001, cluster number=

AnnData object with n_obs × n_vars = 8105 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [13]:
import matplotlib.pyplot as plt

dataset = '8_pbmc_unsorted_10k'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [14]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '8_pbmc_unsorted_10k'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.493298,0.599320,0.718748,0.989912,0.563205,Garfield,8_pbmc_unsorted_10k
1,0.448551,0.562592,0.703769,0.984360,0.535793,Multigrate,8_pbmc_unsorted_10k
2,0.409304,0.635573,0.679667,0.991242,0.579317,MultiVI,8_pbmc_unsorted_10k
3,0.433904,0.610499,0.705069,0.988027,0.574253,MOFA,8_pbmc_unsorted_10k
4,0.428831,0.635575,0.687655,0.990538,0.594528,Seurat,8_pbmc_unsorted_10k


#### 使用默认固定的resolution

In [13]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('8_pbmc_unsorted_10k')
adata

Chunk 81 does not have enough neighbors. Skipping...
Chunk 148 does not have enough neighbors. Skipping...
Chunk 201 does not have enough neighbors. Skipping...
Chunk 227 does not have enough neighbors. Skipping...
Chunk 346 does not have enough neighbors. Skipping...
Chunk 409 does not have enough neighbors. Skipping...
Chunk 538 does not have enough neighbors. Skipping...
Chunk 635 does not have enough neighbors. Skipping...
Chunk 947 does not have enough neighbors. Skipping...
Chunk 1035 does not have enough neighbors. Skipping...
Chunk 1214 does not have enough neighbors. Skipping...
Chunk 1299 does not have enough neighbors. Skipping...
Chunk 1379 does not have enough neighbors. Skipping...
Chunk 1659 does not have enough neighbors. Skipping...
Chunk 1708 does not have enough neighbors. Skipping...
Chunk 1729 does not have enough neighbors. Skipping...
Chunk 1838 does not have enough neighbors. Skipping...
Chunk 2083 does not have enough neighbors. Skipping...
Chunk 2243 does not 

AnnData object with n_obs × n_vars = 8105 × 13000
    obs: 'group', 'cell_type', 'batch', 'cluster', 'silhouette_temp'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [14]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '8_pbmc_unsorted_10k'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_stable_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.387091,0.599320,0.689900,0.989912,0.563205,Garfield,8_pbmc_unsorted_10k
1,0.497786,0.562592,0.713214,0.984360,0.535793,Multigrate,8_pbmc_unsorted_10k
2,0.272640,0.635573,0.641279,0.991242,0.579317,MultiVI,8_pbmc_unsorted_10k
3,0.406799,0.610499,0.690732,0.988027,0.574253,MOFA,8_pbmc_unsorted_10k
4,0.177652,0.635575,0.597946,0.990538,0.594528,Seurat,8_pbmc_unsorted_10k


#### 利用最佳分辨率计算指标(Garfield);

In [8]:
## 记录resolution
reso_dict = {
    '8_pbmc_unsorted_10k': {
        'Garfield': 0.2, #  0.66,
        'Multigrate': 1.750000000000001,
        'MultiVI': 0.45,
        'MOFA': 0.8000000000000003,
        'Seurat': 0.14121476824050008, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('8_pbmc_unsorted_10k', reso_dict=reso_dict)
adata

the length of cell type: 17
method_reso is not None, using the 0.2 as resolution for Garfield...
Number of clusters: 10
the length of cell type: 17
method_reso is not None, using the 1.750000000000001 as resolution for Multigrate...
Number of clusters: 17
the length of cell type: 17
method_reso is not None, using the 0.45 as resolution for MultiVI...
Number of clusters: 17
the length of cell type: 17
method_reso is not None, using the 0.8000000000000003 as resolution for MOFA...
Number of clusters: 17
the length of cell type: 17
method_reso is not None, using the 0.14121476824050008 as resolution for Seurat...
Number of clusters: 17
Chunk 81 does not have enough neighbors. Skipping...
Chunk 148 does not have enough neighbors. Skipping...
Chunk 201 does not have enough neighbors. Skipping...
Chunk 227 does not have enough neighbors. Skipping...
Chunk 346 does not have enough neighbors. Skipping...
Chunk 409 does not have enough neighbors. Skipping...
Chunk 538 does not have enough neigh

AnnData object with n_obs × n_vars = 8105 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [9]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '8_pbmc_unsorted_10k'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.680520,0.599320,0.754724,0.989912,0.563205,Garfield,8_pbmc_unsorted_10k
1,0.448551,0.562592,0.703769,0.984360,0.535793,Multigrate,8_pbmc_unsorted_10k
2,0.409304,0.635573,0.679667,0.991242,0.579317,MultiVI,8_pbmc_unsorted_10k
3,0.433904,0.610499,0.705069,0.988027,0.574253,MOFA,8_pbmc_unsorted_10k
4,0.428831,0.635575,0.687655,0.990538,0.594528,Seurat,8_pbmc_unsorted_10k


### 数据集 7_human_pbmc_10x

In [6]:
## 记录resolution
reso_dict = {
    '7_human_pbmc_10x': {
        'Garfield': 0.405,
        'Multigrate': 1.550000000000001,
        'MultiVI': 0.17433922005000008,
        'MOFA': 0.32805000000000006,
        'Seurat': 0.08338590849833288, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('7_human_pbmc_10x', reso_dict=None)
adata

the length of cell type: 10
Searching resolution for Garfield...
Searching resolution...
resolution=1.9500000000000013, cluster number=23
resolution=1.9000000000000012, cluster number=22
resolution=1.8500000000000012, cluster number=22
resolution=1.8000000000000012, cluster number=22
resolution=1.750000000000001, cluster number=22
resolution=1.700000000000001, cluster number=20
resolution=1.650000000000001, cluster number=20
resolution=1.600000000000001, cluster number=19
resolution=1.550000000000001, cluster number=20
resolution=1.5000000000000009, cluster number=20
resolution=1.4500000000000008, cluster number=20
resolution=1.4000000000000008, cluster number=19
resolution=1.3500000000000008, cluster number=19
resolution=1.3000000000000007, cluster number=19
resolution=1.2500000000000007, cluster number=19
resolution=1.2000000000000006, cluster number=17
resolution=1.1500000000000006, cluster number=16
resolution=1.1000000000000005, cluster number=16
resolution=1.0500000000000005, clu

AnnData object with n_obs × n_vars = 2592 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [7]:
import matplotlib.pyplot as plt

dataset = '7_human_pbmc_10x'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [8]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '7_human_pbmc_10x'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.664271,0.689879,0.768072,0.989808,0.661599,Garfield,7_human_pbmc_10x
1,0.540047,0.565031,0.712995,0.930917,0.568089,Multigrate,7_human_pbmc_10x
2,0.791090,0.711974,0.820206,0.996757,0.648317,MultiVI,7_human_pbmc_10x
3,0.573762,0.678811,0.709448,0.984283,0.648767,MOFA,7_human_pbmc_10x
4,0.654868,0.796970,0.778771,0.996275,0.719270,Seurat,7_human_pbmc_10x


#### 探索一下用最佳分辨率

In [9]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('7_human_pbmc_10x')
adata

resolution: 0.1, nmi: 0.7548250486811503
resolution: 0.2, nmi: 0.7727208318775747
resolution: 0.3, nmi: 0.7766032905563575
resolution: 0.4, nmi: 0.7680720232818008
resolution: 0.5, nmi: 0.7414096358489178
resolution: 0.6, nmi: 0.7412102399731031
resolution: 0.7, nmi: 0.7313968635832362
resolution: 0.8, nmi: 0.7234353599305173
resolution: 0.9, nmi: 0.7126437547504965
resolution: 1.0, nmi: 0.7143406085174687
resolution: 1.1, nmi: 0.7132795202055161
resolution: 1.2, nmi: 0.7084406263403178
resolution: 1.3, nmi: 0.6887413344749832
resolution: 1.4, nmi: 0.6857555295772277
resolution: 1.5, nmi: 0.6824804655781431
resolution: 1.6, nmi: 0.685031497890369
resolution: 1.7, nmi: 0.6849403050743973
resolution: 1.8, nmi: 0.6793120458178192
resolution: 1.9, nmi: 0.6737338164869416
resolution: 2.0, nmi: 0.677368910577878
optimised clustering against cell_type
optimal cluster resolution: 0.3
optimal score: 0.7766032905563575
resolution: 0.1, nmi: 0.6988915760510876
resolution: 0.2, nmi: 0.757697153250

AnnData object with n_obs × n_vars = 2592 × 13000
    obs: 'group', 'cell_type', 'batch', 'silhouette_temp', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [10]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '7_human_pbmc_10x'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.674823,0.689879,0.776603,0.989808,0.661599,Garfield,7_human_pbmc_10x
1,0.679863,0.565031,0.761804,0.930917,0.568089,Multigrate,7_human_pbmc_10x
2,0.782798,0.711974,0.817902,0.996757,0.648317,MultiVI,7_human_pbmc_10x
3,0.630354,0.678811,0.755995,0.984283,0.648767,MOFA,7_human_pbmc_10x
4,0.586445,0.796970,0.763496,0.996275,0.719270,Seurat,7_human_pbmc_10x


#### 利用最佳分辨率计算指标(Garfield);

In [16]:
## 记录resolution
reso_dict = {
    '7_human_pbmc_10x': {
        'Garfield': 0.3, # 0.405,
        'Multigrate': 1.550000000000001,
        'MultiVI': 0.17433922005000008,
        'MOFA': 0.32805000000000006,
        'Seurat': 0.08338590849833288, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('7_human_pbmc_10x', reso_dict=reso_dict)
adata

the length of cell type: 10
method_reso is not None, using the 0.3 as resolution for Garfield...
Number of clusters: 10
the length of cell type: 10
method_reso is not None, using the 1.550000000000001 as resolution for Multigrate...
Number of clusters: 10
the length of cell type: 10
method_reso is not None, using the 0.17433922005000008 as resolution for MultiVI...
Number of clusters: 10
the length of cell type: 10
method_reso is not None, using the 0.32805000000000006 as resolution for MOFA...
Number of clusters: 10
Chunk 26 does not have enough neighbors. Skipping...
Chunk 136 does not have enough neighbors. Skipping...
Chunk 195 does not have enough neighbors. Skipping...
Chunk 216 does not have enough neighbors. Skipping...
Chunk 341 does not have enough neighbors. Skipping...
Chunk 359 does not have enough neighbors. Skipping...
Chunk 452 does not have enough neighbors. Skipping...
Chunk 497 does not have enough neighbors. Skipping...
Chunk 529 does not have enough neighbors. Skip

AnnData object with n_obs × n_vars = 2592 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [17]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '7_human_pbmc_10x'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.674823,0.689879,0.776603,0.989808,0.661599,Garfield,7_human_pbmc_10x
1,0.540047,0.565031,0.712995,0.930917,0.568089,Multigrate,7_human_pbmc_10x
2,0.791090,0.711974,0.820206,0.996757,0.648317,MultiVI,7_human_pbmc_10x
3,0.573762,0.678811,0.709448,0.984283,0.648767,MOFA,7_human_pbmc_10x
4,0.654868,0.796970,0.778771,0.996275,0.719270,Seurat,7_human_pbmc_10x


### 数据集 6_pbmc_granulocyte_sorted_10x

In [15]:
## 记录resolution
reso_dict = {
    '6_pbmc_granulocyte_sorted_10x': {
        'Garfield': 0.6000000000000001,
        'Multigrate': 0.9500000000000004,
        'MultiVI': 0.47,
        'MOFA': 0.6500000000000001,
        'Seurat': 0.0926510094425921, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('6_pbmc_granulocyte_sorted_10x', reso_dict=reso_dict)
adata

the length of cell type: 17
method_reso is not None, using the 0.6000000000000001 as resolution for Garfield...
Number of clusters: 17
the length of cell type: 17
method_reso is not None, using the 0.9500000000000004 as resolution for Multigrate...
Number of clusters: 17
the length of cell type: 17
method_reso is not None, using the 0.47 as resolution for MultiVI...
Number of clusters: 17
the length of cell type: 17
Searching resolution for MOFA...
Searching resolution...
resolution=1.9500000000000013, cluster number=36
resolution=1.9000000000000012, cluster number=34
resolution=1.8500000000000012, cluster number=34
resolution=1.8000000000000012, cluster number=31
resolution=1.750000000000001, cluster number=32
resolution=1.700000000000001, cluster number=31
resolution=1.650000000000001, cluster number=29
resolution=1.600000000000001, cluster number=31
resolution=1.550000000000001, cluster number=28
resolution=1.5000000000000009, cluster number=30
resolution=1.4500000000000008, cluster

AnnData object with n_obs × n_vars = 10137 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [16]:
import matplotlib.pyplot as plt

dataset = '6_pbmc_granulocyte_sorted_10x'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [17]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '6_pbmc_granulocyte_sorted_10x'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.554142,0.648922,0.751475,0.992975,0.602007,Garfield,6_pbmc_granulocyte_sorted_10x
1,0.533804,0.597531,0.717887,0.986449,0.549878,Multigrate,6_pbmc_granulocyte_sorted_10x
2,0.426603,0.645374,0.689509,0.992112,0.574815,MultiVI,6_pbmc_granulocyte_sorted_10x
3,0.492799,0.627424,0.715005,0.990376,0.569079,MOFA,6_pbmc_granulocyte_sorted_10x
4,0.581648,0.676489,0.731705,0.992321,0.600173,Seurat,6_pbmc_granulocyte_sorted_10x


#### 探索一下用最佳分辨率

In [20]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('6_pbmc_granulocyte_sorted_10x')
adata

resolution: 0.1, nmi: 0.7626309050410545
resolution: 0.2, nmi: 0.7757976702108258
resolution: 0.3, nmi: 0.7671757649936511
resolution: 0.4, nmi: 0.7529705870286229
resolution: 0.5, nmi: 0.7663532958988659
resolution: 0.6, nmi: 0.7514754169419015
resolution: 0.7, nmi: 0.7285444394906712
resolution: 0.8, nmi: 0.7200502175502791
resolution: 0.9, nmi: 0.7051785602148432
resolution: 1.0, nmi: 0.711517867495174
resolution: 1.1, nmi: 0.7038227577420872
resolution: 1.2, nmi: 0.6970653119746127
resolution: 1.3, nmi: 0.6890439920006236
resolution: 1.4, nmi: 0.6845148520241829
resolution: 1.5, nmi: 0.6781082407926855
resolution: 1.6, nmi: 0.6783780952730941
resolution: 1.7, nmi: 0.6693900209083833
resolution: 1.8, nmi: 0.6685127042791499
resolution: 1.9, nmi: 0.6576845767325323
resolution: 2.0, nmi: 0.657188554631055
optimised clustering against cell_type
optimal cluster resolution: 0.2
optimal score: 0.7757976702108258
resolution: 0.1, nmi: 0.7056741548743519
resolution: 0.2, nmi: 0.760279055269

AnnData object with n_obs × n_vars = 10137 × 13000
    obs: 'group', 'cell_type', 'batch', 'silhouette_temp', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [6]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '6_pbmc_granulocyte_sorted_10x'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.748856,0.648922,0.775798,0.992975,0.602007,Garfield,6_pbmc_granulocyte_sorted_10x
1,0.752448,0.597531,0.773145,0.986449,0.549878,Multigrate,6_pbmc_granulocyte_sorted_10x
2,0.748852,0.645374,0.771078,0.992112,0.574815,MultiVI,6_pbmc_granulocyte_sorted_10x
3,0.647237,0.627424,0.770455,0.990376,0.569079,MOFA,6_pbmc_granulocyte_sorted_10x
4,0.548936,0.676489,0.723240,0.992321,0.600173,Seurat,6_pbmc_granulocyte_sorted_10x


#### 利用最佳分辨率计算指标(Garfield);

In [18]:
## 记录resolution
reso_dict = {
    '6_pbmc_granulocyte_sorted_10x': {
        'Garfield': 0.2, # 0.6000000000000001,
        'Multigrate': 0.9500000000000004,
        'MultiVI': 0.47,
        'MOFA': 0.6500000000000001,
        'Seurat': 0.0926510094425921, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('6_pbmc_granulocyte_sorted_10x', reso_dict=reso_dict)
adata

the length of cell type: 17
method_reso is not None, using the 0.2 as resolution for Garfield...
Number of clusters: 9
the length of cell type: 17
method_reso is not None, using the 0.9500000000000004 as resolution for Multigrate...
Number of clusters: 17
the length of cell type: 17
method_reso is not None, using the 0.47 as resolution for MultiVI...
Number of clusters: 17
the length of cell type: 17
method_reso is not None, using the 0.6500000000000001 as resolution for MOFA...
Number of clusters: 17
the length of cell type: 17
method_reso is not None, using the 0.0926510094425921 as resolution for Seurat...
Number of clusters: 17
6_pbmc_granulocyte_sorted_10x


AnnData object with n_obs × n_vars = 10137 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [19]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '6_pbmc_granulocyte_sorted_10x'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.748856,0.648922,0.775798,0.992975,0.602007,Garfield,6_pbmc_granulocyte_sorted_10x
1,0.533804,0.597531,0.717887,0.986449,0.549878,Multigrate,6_pbmc_granulocyte_sorted_10x
2,0.426603,0.645374,0.689509,0.992112,0.574815,MultiVI,6_pbmc_granulocyte_sorted_10x
3,0.492799,0.627424,0.715005,0.990376,0.569079,MOFA,6_pbmc_granulocyte_sorted_10x
4,0.581648,0.676489,0.731705,0.992321,0.600173,Seurat,6_pbmc_granulocyte_sorted_10x


### 数据集 5_human_brain_10x

In [11]:
## 记录resolution
reso_dict = {
    '5_human_brain_10x': {
        'Garfield': 0.245,
        'Multigrate': 0.7500000000000002,
        'MultiVI': 0.0750473176484996,
        'MOFA': 0.17709329141645008,
        'Seurat': 0.021195579137608122, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('5_human_brain_10x', reso_dict=reso_dict)
adata

the length of cell type: 8
method_reso is not None, using the 0.245 as resolution for Garfield...
Number of clusters: 9
the length of cell type: 8
method_reso is not None, using the 0.7500000000000002 as resolution for Multigrate...
Number of clusters: 8
the length of cell type: 8
method_reso is not None, using the 0.0750473176484996 as resolution for MultiVI...
Number of clusters: 8
the length of cell type: 8
method_reso is not None, using the 0.17709329141645008 as resolution for MOFA...
Number of clusters: 8
the length of cell type: 8
method_reso is not None, using the 0.021195579137608122 as resolution for Seurat...
Number of clusters: 8
Chunk 126 does not have enough neighbors. Skipping...
Chunk 218 does not have enough neighbors. Skipping...
Chunk 378 does not have enough neighbors. Skipping...
Chunk 606 does not have enough neighbors. Skipping...
Chunk 851 does not have enough neighbors. Skipping...
Chunk 1175 does not have enough neighbors. Skipping...
Chunk 1326 does not have 

AnnData object with n_obs × n_vars = 2855 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [12]:
import matplotlib.pyplot as plt

dataset = '5_human_brain_10x'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [13]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '5_human_brain_10x'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.604645,0.715671,0.807935,1.0,0.825768,Garfield,5_human_brain_10x
1,0.576274,0.606063,0.723754,1.0,0.741968,Multigrate,5_human_brain_10x
2,0.547282,0.718769,0.713029,1.0,0.722927,MultiVI,5_human_brain_10x
3,0.576093,0.707015,0.775741,1.0,0.760355,MOFA,5_human_brain_10x
4,0.674943,0.812406,0.820338,1.0,0.789231,Seurat,5_human_brain_10x


#### 探索一下用最佳分辨率

In [7]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('5_human_brain_10x')
adata

resolution: 0.1, nmi: 0.7829232729989649
resolution: 0.2, nmi: 0.8013067242474287
resolution: 0.3, nmi: 0.7309640391062733
resolution: 0.4, nmi: 0.7207422629546304
resolution: 0.5, nmi: 0.7261748581681903
resolution: 0.6, nmi: 0.6942114506851838
resolution: 0.7, nmi: 0.6695791534759318
resolution: 0.8, nmi: 0.6427437823971089
resolution: 0.9, nmi: 0.6364089273473252
resolution: 1.0, nmi: 0.6207374905333479
resolution: 1.1, nmi: 0.6311015513479621
resolution: 1.2, nmi: 0.609930130253816
resolution: 1.3, nmi: 0.6122000955115091
resolution: 1.4, nmi: 0.6049432765477988
resolution: 1.5, nmi: 0.5967870520318065
resolution: 1.6, nmi: 0.5888017749026527
resolution: 1.7, nmi: 0.5803193157058757
resolution: 1.8, nmi: 0.5721182149858675
resolution: 1.9, nmi: 0.5671236964986385
resolution: 2.0, nmi: 0.5645492917793165
optimised clustering against cell_type
optimal cluster resolution: 0.2
optimal score: 0.8013067242474287
resolution: 0.1, nmi: 0.7745229101914064
resolution: 0.2, nmi: 0.79305795597

AnnData object with n_obs × n_vars = 2855 × 13000
    obs: 'group', 'cell_type', 'batch', 'silhouette_temp', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [9]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '5_human_brain_10x'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.610099,0.715671,0.801307,1.0,0.825768,Garfield,5_human_brain_10x
1,0.925922,0.606063,0.793058,1.0,0.741968,Multigrate,5_human_brain_10x
2,0.390442,0.718769,0.664225,1.0,0.722927,MultiVI,5_human_brain_10x
3,0.535730,0.707015,0.752115,1.0,0.760355,MOFA,5_human_brain_10x
4,0.340883,0.812406,0.690390,1.0,0.789231,Seurat,5_human_brain_10x


#### 利用最佳分辨率计算指标(Garfield);

In [20]:
## 记录resolution
reso_dict = {
    '5_human_brain_10x': {
        'Garfield': 0.2, # 0.245,
        'Multigrate': 0.7500000000000002,
        'MultiVI': 0.0750473176484996,
        'MOFA': 0.17709329141645008,
        'Seurat': 0.021195579137608122, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('5_human_brain_10x', reso_dict=reso_dict)
adata

the length of cell type: 8
method_reso is not None, using the 0.2 as resolution for Garfield...
Number of clusters: 7
the length of cell type: 8
method_reso is not None, using the 0.7500000000000002 as resolution for Multigrate...
Number of clusters: 8
the length of cell type: 8
method_reso is not None, using the 0.0750473176484996 as resolution for MultiVI...
Number of clusters: 8
the length of cell type: 8
method_reso is not None, using the 0.17709329141645008 as resolution for MOFA...
Number of clusters: 8
the length of cell type: 8
method_reso is not None, using the 0.021195579137608122 as resolution for Seurat...
Number of clusters: 8
Chunk 126 does not have enough neighbors. Skipping...
Chunk 218 does not have enough neighbors. Skipping...
Chunk 378 does not have enough neighbors. Skipping...
Chunk 606 does not have enough neighbors. Skipping...
Chunk 851 does not have enough neighbors. Skipping...
Chunk 1175 does not have enough neighbors. Skipping...
Chunk 1326 does not have en

AnnData object with n_obs × n_vars = 2855 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [21]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '5_human_brain_10x'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.610099,0.715671,0.801307,1.0,0.825768,Garfield,5_human_brain_10x
1,0.576274,0.606063,0.723754,1.0,0.741968,Multigrate,5_human_brain_10x
2,0.547282,0.718769,0.713029,1.0,0.722927,MultiVI,5_human_brain_10x
3,0.576093,0.707015,0.775741,1.0,0.760355,MOFA,5_human_brain_10x
4,0.674943,0.812406,0.820338,1.0,0.789231,Seurat,5_human_brain_10x


### 数据集 4_brain_ISSAAC_seq

In [24]:
## 记录resolution
reso_dict = {
    '4_brain_ISSAAC_seq': {
        'Garfield': 0.8800000000000003,
        'Multigrate': 1.2400000000000007,
        'MultiVI': 0.8300000000000003,
        'MOFA': 0.9200000000000004,
        'Seurat': 0.11438396227480506, #0.11438396227480506
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('4_brain_ISSAAC_seq', reso_dict=reso_dict, 
                                     start=0.5, end=0.75, increment=0.05)
adata

the length of cell type: 20
method_reso is not None, using the 0.8800000000000003 as resolution for Garfield...
Number of clusters: 20
the length of cell type: 20
method_reso is not None, using the 1.2400000000000007 as resolution for Multigrate...
Number of clusters: 20
the length of cell type: 20
method_reso is not None, using the 0.8300000000000003 as resolution for MultiVI...
Number of clusters: 20
the length of cell type: 20
method_reso is not None, using the 0.9200000000000004 as resolution for MOFA...
Number of clusters: 20
the length of cell type: 20
Searching resolution for Seurat...
Searching resolution...
resolution=0.7000000000000002, cluster number=47
resolution=0.6500000000000001, cluster number=45
resolution=0.6000000000000001, cluster number=44
resolution=0.55, cluster number=42
resolution=0.5, cluster number=40
Expanding search range: new range (0.45, 0.7875)
resolution=0.75, cluster number=49
resolution=0.7, cluster number=47
resolution=0.6499999999999999, cluster num

AnnData object with n_obs × n_vars = 10361 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [25]:
import matplotlib.pyplot as plt

dataset = '4_brain_ISSAAC_seq'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [26]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '4_brain_ISSAAC_seq'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.355023,0.511901,0.530824,0.967721,0.478869,Garfield,4_brain_ISSAAC_seq
1,0.350469,0.513112,0.540713,0.954077,0.498653,Multigrate,4_brain_ISSAAC_seq
2,0.408146,0.521162,0.549681,0.971718,0.483725,MultiVI,4_brain_ISSAAC_seq
3,0.322902,0.500459,0.491447,0.959406,0.487024,MOFA,4_brain_ISSAAC_seq
4,0.403492,0.518850,0.580678,0.983473,0.401643,Seurat,4_brain_ISSAAC_seq


#### 探索一下用最佳分辨率

In [6]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('4_brain_ISSAAC_seq')
adata

resolution: 0.1, nmi: 0.4679116587409615
resolution: 0.2, nmi: 0.46953868634534324
resolution: 0.3, nmi: 0.5073591774085828
resolution: 0.4, nmi: 0.5274331706107964
resolution: 0.5, nmi: 0.5290364941284094
resolution: 0.6, nmi: 0.5241942609308612
resolution: 0.7, nmi: 0.5356521258957119
resolution: 0.8, nmi: 0.5308954787785267
resolution: 0.9, nmi: 0.5256782772555346
resolution: 1.0, nmi: 0.5233423755749724
resolution: 1.1, nmi: 0.5127526275980276
resolution: 1.2, nmi: 0.5161950286110775
resolution: 1.3, nmi: 0.5227922374428045
resolution: 1.4, nmi: 0.517237063847293
resolution: 1.5, nmi: 0.5138358942781424
resolution: 1.6, nmi: 0.509028087937334
resolution: 1.7, nmi: 0.5211717453501659
resolution: 1.8, nmi: 0.5152012781627139
resolution: 1.9, nmi: 0.5131925501842107
resolution: 2.0, nmi: 0.5141684337350988
optimised clustering against cell_type
optimal cluster resolution: 0.7
optimal score: 0.5356521258957119
resolution: 0.1, nmi: 0.37370868028359755
resolution: 0.2, nmi: 0.4655026794

AnnData object with n_obs × n_vars = 10361 × 13000
    obs: 'group', 'cell_type', 'batch', 'silhouette_temp', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [7]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '4_brain_ISSAAC_seq'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.383082,0.511901,0.535652,0.967721,0.478869,Garfield,4_brain_ISSAAC_seq
1,0.482315,0.513112,0.558956,0.954077,0.498653,Multigrate,4_brain_ISSAAC_seq
2,0.478640,0.521162,0.563583,0.971718,0.483725,MultiVI,4_brain_ISSAAC_seq
3,0.374167,0.500459,0.497306,0.959406,0.487024,MOFA,4_brain_ISSAAC_seq
4,0.429688,0.518850,0.583435,0.983473,0.401643,Seurat,4_brain_ISSAAC_seq


#### 利用最佳分辨率计算指标(Garfield);

In [22]:
## 记录resolution
reso_dict = {
    '4_brain_ISSAAC_seq': {
        'Garfield': 0.7, # 0.8800000000000003,
        'Multigrate': 1.2400000000000007,
        'MultiVI': 0.8300000000000003,
        'MOFA': 0.9200000000000004,
        'Seurat': 0.11438396227480506, #0.11438396227480506
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('4_brain_ISSAAC_seq', reso_dict=reso_dict, 
                                     start=0.5, end=0.75, increment=0.05)
adata

the length of cell type: 20
method_reso is not None, using the 0.7 as resolution for Garfield...
Number of clusters: 18
the length of cell type: 20
method_reso is not None, using the 1.2400000000000007 as resolution for Multigrate...
Number of clusters: 20
the length of cell type: 20
method_reso is not None, using the 0.8300000000000003 as resolution for MultiVI...
Number of clusters: 20
the length of cell type: 20
method_reso is not None, using the 0.9200000000000004 as resolution for MOFA...
Number of clusters: 20
the length of cell type: 20
method_reso is not None, using the 0.11438396227480506 as resolution for Seurat...
Number of clusters: 20
Chunk 766 does not have enough neighbors. Skipping...
Chunk 1157 does not have enough neighbors. Skipping...
Chunk 1459 does not have enough neighbors. Skipping...
Chunk 2504 does not have enough neighbors. Skipping...
Chunk 3266 does not have enough neighbors. Skipping...
Chunk 3631 does not have enough neighbors. Skipping...
Chunk 3643 does

AnnData object with n_obs × n_vars = 10361 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [23]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '4_brain_ISSAAC_seq'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.383082,0.511901,0.535652,0.967721,0.478869,Garfield,4_brain_ISSAAC_seq
1,0.350469,0.513112,0.540713,0.954077,0.498653,Multigrate,4_brain_ISSAAC_seq
2,0.408146,0.521162,0.549681,0.971718,0.483725,MultiVI,4_brain_ISSAAC_seq
3,0.322902,0.500459,0.491447,0.959406,0.487024,MOFA,4_brain_ISSAAC_seq
4,0.403492,0.518850,0.580678,0.983473,0.401643,Seurat,4_brain_ISSAAC_seq


### 数据集 3_brain_SNARE

In [16]:
## 记录resolution
reso_dict = {
    '3_brain_SNARE': {
        'Garfield': 0.6500000000000001,
        'Multigrate': 1.4500000000000008,
        'MultiVI': 0.2952450000000001,
        'MOFA': 0.5,
        'Seurat': 0.10294556604732455,
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('3_brain_SNARE', reso_dict=reso_dict)
adata

the length of cell type: 13
method_reso is not None, using the 0.6500000000000001 as resolution for Garfield...
Number of clusters: 13
the length of cell type: 13
method_reso is not None, using the 1.4500000000000008 as resolution for Multigrate...
Number of clusters: 13
the length of cell type: 13
method_reso is not None, using the 0.2952450000000001 as resolution for MultiVI...
Number of clusters: 13
the length of cell type: 13
method_reso is not None, using the 0.5 as resolution for MOFA...
Number of clusters: 13
the length of cell type: 13
Searching resolution for Seurat...
Searching resolution...
resolution=1.9500000000000013, cluster number=68
resolution=1.9000000000000012, cluster number=65
resolution=1.8500000000000012, cluster number=63
resolution=1.8000000000000012, cluster number=62
resolution=1.750000000000001, cluster number=61
resolution=1.700000000000001, cluster number=61
resolution=1.650000000000001, cluster number=60
resolution=1.600000000000001, cluster number=60
res

AnnData object with n_obs × n_vars = 8055 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [17]:
import matplotlib.pyplot as plt

dataset = '3_brain_SNARE'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [19]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '3_brain_SNARE'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.568369,0.616415,0.721559,0.993765,0.606543,Garfield,3_brain_SNARE
1,0.388010,0.540925,0.582473,0.962407,0.535359,Multigrate,3_brain_SNARE
2,0.387346,0.604648,0.587641,0.989677,0.548244,MultiVI,3_brain_SNARE
3,0.416283,0.525512,0.616441,0.988326,0.562822,MOFA,3_brain_SNARE
4,0.578907,0.786710,0.745991,0.998353,0.707169,Seurat,3_brain_SNARE


#### 探索一下用最佳分辨率

In [8]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('3_brain_SNARE')
adata

resolution: 0.1, nmi: 0.6792656140943156
resolution: 0.2, nmi: 0.7108192605432453
resolution: 0.3, nmi: 0.7265615194043833
resolution: 0.4, nmi: 0.7394168432821027
resolution: 0.5, nmi: 0.7232830381888907
resolution: 0.6, nmi: 0.7320908320188952
resolution: 0.7, nmi: 0.7018328715018829
resolution: 0.8, nmi: 0.6845877370400282
resolution: 0.9, nmi: 0.683113660570257
resolution: 1.0, nmi: 0.6794504956749007
resolution: 1.1, nmi: 0.6770522886299539
resolution: 1.2, nmi: 0.6607830254277536
resolution: 1.3, nmi: 0.6607373415349395
resolution: 1.4, nmi: 0.6482328063061619
resolution: 1.5, nmi: 0.652008213753001
resolution: 1.6, nmi: 0.6484879455354954
resolution: 1.7, nmi: 0.6421130323509258
resolution: 1.8, nmi: 0.624447532980517
resolution: 1.9, nmi: 0.6166676990694565
resolution: 2.0, nmi: 0.6112489270687637
optimised clustering against cell_type
optimal cluster resolution: 0.4
optimal score: 0.7394168432821027
resolution: 0.1, nmi: 0.5065707027914722
resolution: 0.2, nmi: 0.6453234949416

AnnData object with n_obs × n_vars = 8055 × 13000
    obs: 'group', 'cell_type', 'batch', 'silhouette_temp', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [9]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '3_brain_SNARE'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.678792,0.616415,0.739417,0.993765,0.606543,Garfield,3_brain_SNARE
1,0.582377,0.540925,0.672762,0.962407,0.535359,Multigrate,3_brain_SNARE
2,0.544277,0.604648,0.617965,0.989677,0.548244,MultiVI,3_brain_SNARE
3,0.600631,0.525512,0.686458,0.988326,0.562822,MOFA,3_brain_SNARE
4,0.578907,0.786710,0.745991,0.998353,0.707169,Seurat,3_brain_SNARE


#### 利用最佳分辨率计算指标(Garfield);

In [24]:
## 记录resolution
reso_dict = {
    '3_brain_SNARE': {
        'Garfield': 0.4, # 0.6500000000000001,
        'Multigrate': 1.4500000000000008,
        'MultiVI': 0.2952450000000001,
        'MOFA': 0.5,
        'Seurat': 0.10294556604732455,
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('3_brain_SNARE', reso_dict=reso_dict)
adata

the length of cell type: 13
method_reso is not None, using the 0.4 as resolution for Garfield...
Number of clusters: 11
the length of cell type: 13
method_reso is not None, using the 1.4500000000000008 as resolution for Multigrate...
Number of clusters: 13
the length of cell type: 13
method_reso is not None, using the 0.2952450000000001 as resolution for MultiVI...
Number of clusters: 13
the length of cell type: 13
method_reso is not None, using the 0.5 as resolution for MOFA...
Number of clusters: 13
the length of cell type: 13
method_reso is not None, using the 0.10294556604732455 as resolution for Seurat...
Number of clusters: 13
3_brain_SNARE


AnnData object with n_obs × n_vars = 8055 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [25]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '3_brain_SNARE'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.678792,0.616415,0.739417,0.993765,0.606543,Garfield,3_brain_SNARE
1,0.388010,0.540925,0.582473,0.962407,0.535359,Multigrate,3_brain_SNARE
2,0.387346,0.604648,0.587641,0.989677,0.548244,MultiVI,3_brain_SNARE
3,0.416283,0.525512,0.616441,0.988326,0.562822,MOFA,3_brain_SNARE
4,0.578907,0.786710,0.745991,0.998353,0.707169,Seurat,3_brain_SNARE


### 数据集 2_brain_ShareSeq

In [ ]:
# resolution: 'Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
# resolution=0.8000000000000003, cluster number=13
# resolution=1.600000000000001, cluster number=13
# resolution=0.39524500000000007, cluster number=13
# resolution=0.6000000000000001, cluster number=13
# resolution for Seurat: 0.44999999999999996

reso_dict = {
    '2_brain_ShareSeq': {
        'Garfield': 0.8000000000000003,
        'Multigrate': 1.600000000000001,
        'MultiVI': 0.39524500000000007,
        'MOFA': 0.6000000000000001,
        'Seurat': 0.44999999999999996
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('2_brain_ShareSeq', reso_dict=None)
adata

In [21]:
## 重复结果
reso_dict = {
    '2_brain_ShareSeq': {
        'Garfield': 0.8000000000000003,
        'Multigrate': 1.600000000000001,
        'MultiVI': 0.39524500000000007,
        'MOFA': 0.6000000000000001,
        'Seurat': 0.10294556604732455
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('2_brain_ShareSeq', reso_dict)
adata

the length of cell type: 13
method_reso is not None, using the 0.8000000000000003 as resolution for Garfield...
Number of clusters: 13
the length of cell type: 13
method_reso is not None, using the 1.600000000000001 as resolution for Multigrate...
Number of clusters: 13
the length of cell type: 13
method_reso is not None, using the 0.39524500000000007 as resolution for MultiVI...
Number of clusters: 13
the length of cell type: 13
method_reso is not None, using the 0.6000000000000001 as resolution for MOFA...
Number of clusters: 13
Chunk 8 does not have enough neighbors. Skipping...
Chunk 14 does not have enough neighbors. Skipping...
Chunk 157 does not have enough neighbors. Skipping...
Chunk 165 does not have enough neighbors. Skipping...
Chunk 206 does not have enough neighbors. Skipping...
Chunk 237 does not have enough neighbors. Skipping...
Chunk 251 does not have enough neighbors. Skipping...
Chunk 310 does not have enough neighbors. Skipping...
Chunk 396 does not have enough nei

AnnData object with n_obs × n_vars = 2344 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [22]:
import matplotlib.pyplot as plt

dataset = '2_brain_ShareSeq'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [23]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '2_brain_ShareSeq'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.400663,0.514628,0.589198,0.922453,0.495120,Garfield,2_brain_ShareSeq
1,0.174423,0.480161,0.378802,0.836010,0.474992,Multigrate,2_brain_ShareSeq
2,0.458934,0.643207,0.638186,0.970894,0.600148,MultiVI,2_brain_ShareSeq
3,0.451449,0.583920,0.639585,0.957829,0.560575,MOFA,2_brain_ShareSeq
4,0.465474,0.667899,0.647241,0.994003,0.604726,Seurat,2_brain_ShareSeq


#### 探索一下用最佳分辨率

In [10]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('2_brain_ShareSeq')
adata

resolution: 0.1, nmi: 0.42485459441957824
resolution: 0.2, nmi: 0.44297030507512464
resolution: 0.3, nmi: 0.5597662822684231
resolution: 0.4, nmi: 0.5045705935527509
resolution: 0.5, nmi: 0.5385360959021275
resolution: 0.6, nmi: 0.5597239391408851
resolution: 0.7, nmi: 0.5904612922311091
resolution: 0.8, nmi: 0.5891979594702375
resolution: 0.9, nmi: 0.5756272687504639
resolution: 1.0, nmi: 0.5848581661835602
resolution: 1.1, nmi: 0.5783837858075326
resolution: 1.2, nmi: 0.578370715418351
resolution: 1.3, nmi: 0.5814864955208765
resolution: 1.4, nmi: 0.5794902345302025
resolution: 1.5, nmi: 0.575751826768275
resolution: 1.6, nmi: 0.570890264579799
resolution: 1.7, nmi: 0.5694330542582511
resolution: 1.8, nmi: 0.577560478424967
resolution: 1.9, nmi: 0.5741297546744633
resolution: 2.0, nmi: 0.5678927011343805
optimised clustering against cell_type
optimal cluster resolution: 0.7
optimal score: 0.5904612922311091
resolution: 0.1, nmi: 0.3584263741322073
resolution: 0.2, nmi: 0.388185151459

AnnData object with n_obs × n_vars = 2344 × 13000
    obs: 'group', 'cell_type', 'batch', 'silhouette_temp', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [11]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '2_brain_ShareSeq'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.420283,0.514628,0.590461,0.922453,0.495120,Garfield,2_brain_ShareSeq
1,0.303512,0.480161,0.439557,0.836010,0.474992,Multigrate,2_brain_ShareSeq
2,0.506942,0.643207,0.649360,0.970894,0.600148,MultiVI,2_brain_ShareSeq
3,0.511619,0.583920,0.667331,0.957829,0.560575,MOFA,2_brain_ShareSeq
4,0.521676,0.667899,0.689285,0.994003,0.604726,Seurat,2_brain_ShareSeq


#### 利用最佳分辨率计算指标(Garfield);

In [26]:
## 重复结果
reso_dict = {
    '2_brain_ShareSeq': {
        'Garfield': 0.7, # 0.8000000000000003,
        'Multigrate': 1.600000000000001,
        'MultiVI': 0.39524500000000007,
        'MOFA': 0.6000000000000001,
        'Seurat': 0.10294556604732455
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('2_brain_ShareSeq', reso_dict)
adata

the length of cell type: 13
method_reso is not None, using the 0.7 as resolution for Garfield...
Number of clusters: 12
the length of cell type: 13
method_reso is not None, using the 1.600000000000001 as resolution for Multigrate...
Number of clusters: 13
the length of cell type: 13
method_reso is not None, using the 0.39524500000000007 as resolution for MultiVI...
Number of clusters: 13
the length of cell type: 13
method_reso is not None, using the 0.6000000000000001 as resolution for MOFA...
Number of clusters: 13
Chunk 8 does not have enough neighbors. Skipping...
Chunk 14 does not have enough neighbors. Skipping...
Chunk 157 does not have enough neighbors. Skipping...
Chunk 165 does not have enough neighbors. Skipping...
Chunk 206 does not have enough neighbors. Skipping...
Chunk 237 does not have enough neighbors. Skipping...
Chunk 251 does not have enough neighbors. Skipping...
Chunk 310 does not have enough neighbors. Skipping...
Chunk 396 does not have enough neighbors. Skippin

AnnData object with n_obs × n_vars = 2344 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [27]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '2_brain_ShareSeq'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.420283,0.514628,0.590461,0.922453,0.495120,Garfield,2_brain_ShareSeq
1,0.174423,0.480161,0.378802,0.836010,0.474992,Multigrate,2_brain_ShareSeq
2,0.458934,0.643207,0.638186,0.970894,0.600148,MultiVI,2_brain_ShareSeq
3,0.451449,0.583920,0.639585,0.957829,0.560575,MOFA,2_brain_ShareSeq
4,0.465474,0.667899,0.647241,0.994003,0.604726,Seurat,2_brain_ShareSeq


### 数据集 1_ShareSeq_Skin

In [12]:
reso_dict = {
    '1_ShareSeq_Skin': {
        'Garfield': 1.02,
        'Multigrate': 1.9500000000000013,
        'MultiVI': 0.36450000000000005,
        'MOFA': 1.075,
        'Seurat': 0.095
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('1_ShareSeq_Skin', reso_dict=reso_dict)
adata

the length of cell type: 23
method_reso is not None, using the 1.02 as resolution for Garfield...
Number of clusters: 23
the length of cell type: 23
method_reso is not None, using the 1.9500000000000013 as resolution for Multigrate...
Number of clusters: 23
the length of cell type: 23
method_reso is not None, using the 0.36450000000000005 as resolution for MultiVI...
Number of clusters: 23
the length of cell type: 23
method_reso is not None, using the 1.075 as resolution for MOFA...
Number of clusters: 23
the length of cell type: 23
method_reso is not None, using the 0.095 as resolution for Seurat...
Number of clusters: 23
1_ShareSeq_Skin


AnnData object with n_obs × n_vars = 34774 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [15]:
import matplotlib.pyplot as plt

dataset = '1_ShareSeq_Skin'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 4800x1200 with 0 Axes>

<Figure size 4800x1200 with 0 Axes>

<Figure size 4800x1200 with 0 Axes>

<Figure size 4800x1200 with 0 Axes>

<Figure size 4800x1200 with 0 Axes>

In [14]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '1_ShareSeq_Skin'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.376443,0.549971,0.598558,0.980681,0.538438,Garfield,1_ShareSeq_Skin
1,0.395601,0.537026,0.573400,0.956081,0.516575,Multigrate,1_ShareSeq_Skin
2,0.334506,0.587817,0.599579,0.986347,0.556281,MultiVI,1_ShareSeq_Skin
3,0.312498,0.513707,0.532514,0.963540,0.523137,MOFA,1_ShareSeq_Skin
4,0.411604,0.615279,0.629103,0.984065,0.566668,Seurat,1_ShareSeq_Skin


#### 探索一下用最佳分辨率

In [6]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('1_ShareSeq_Skin')
adata


resolution: 0.1, nmi: 0.5353436267963937
resolution: 0.2, nmi: 0.5723102416540203
resolution: 0.3, nmi: 0.6031911869546472
resolution: 0.4, nmi: 0.6165769100045531
resolution: 0.5, nmi: 0.6248911567658882
resolution: 0.6, nmi: 0.623292641896834
resolution: 0.7, nmi: 0.6153468786706748
resolution: 0.8, nmi: 0.615155745396997
resolution: 0.9, nmi: 0.605526028804271
resolution: 1.0, nmi: 0.6035508796503326
resolution: 1.1, nmi: 0.6080362857638141
resolution: 1.2, nmi: 0.5877809469580292
resolution: 1.3, nmi: 0.5912423207636393
resolution: 1.4, nmi: 0.5902486125061525
resolution: 1.5, nmi: 0.5861233158384564
resolution: 1.6, nmi: 0.5845231058141975
resolution: 1.7, nmi: 0.5807327071753347
resolution: 1.8, nmi: 0.5763597794648074
resolution: 1.9, nmi: 0.5763388392068862
resolution: 2.0, nmi: 0.5739639597898369
optimised clustering against cell_type
optimal cluster resolution: 0.5
optimal score: 0.6248911567658882
resolution: 0.1, nmi: 0.19967273312491704
resolution: 0.2, nmi: 0.498204013452

AnnData object with n_obs × n_vars = 34774 × 13000
    obs: 'group', 'cell_type', 'batch', 'silhouette_temp', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [7]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '1_ShareSeq_Skin'

df = pd. read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.459732,0.549971,0.624891,0.980681,0.538438,Garfield,1_ShareSeq_Skin
1,0.425820,0.537026,0.588454,0.956081,0.516575,Multigrate,1_ShareSeq_Skin
2,0.386366,0.587817,0.613151,0.986347,0.556281,MultiVI,1_ShareSeq_Skin
3,0.435989,0.513707,0.560073,0.963540,0.523137,MOFA,1_ShareSeq_Skin
4,0.374092,0.615279,0.613303,0.984065,0.566668,Seurat,1_ShareSeq_Skin


#### 利用最佳分辨率计算指标(Garfield);

In [28]:
reso_dict = {
    '1_ShareSeq_Skin': {
        'Garfield': 0.5, # 1.02,
        'Multigrate': 1.9500000000000013,
        'MultiVI': 0.36450000000000005,
        'MOFA': 1.075,
        'Seurat': 0.095
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('1_ShareSeq_Skin', reso_dict=reso_dict)
adata

the length of cell type: 23
method_reso is not None, using the 0.5 as resolution for Garfield...
Number of clusters: 15
the length of cell type: 23
method_reso is not None, using the 1.9500000000000013 as resolution for Multigrate...
Number of clusters: 23
the length of cell type: 23
method_reso is not None, using the 0.36450000000000005 as resolution for MultiVI...
Number of clusters: 23
the length of cell type: 23
method_reso is not None, using the 1.075 as resolution for MOFA...
Number of clusters: 23
the length of cell type: 23
method_reso is not None, using the 0.095 as resolution for Seurat...
Number of clusters: 23
1_ShareSeq_Skin


AnnData object with n_obs × n_vars = 34774 × 13000
    obs: 'group', 'cell_type', 'batch', 'Garfield_cluster', 'silhouette_temp', 'Multigrate_cluster', 'MultiVI_cluster', 'MOFA_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'MultiVI', 'MultiVI_cluster', 'MOFA', 'MOFA_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'MultiVI', 'MOFA', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'MultiVI_distances', 'MultiVI_connectivities', 'MOFA_distances', 'MOFA_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [29]:
method_list = ['Garfield', 'Multigrate','MultiVI','MOFA','Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_ATAC'
dataset = '1_ShareSeq_Skin'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df

,ARI,Isolated_Labels_ASW,NMI,cLISI_Graph,Silhouette,method,Dataset
0,0.459732,0.549971,0.624891,0.980681,0.538438,Garfield,1_ShareSeq_Skin
1,0.395601,0.537026,0.573400,0.956081,0.516575,Multigrate,1_ShareSeq_Skin
2,0.334506,0.587817,0.599579,0.986347,0.556281,MultiVI,1_ShareSeq_Skin
3,0.312498,0.513707,0.532514,0.963540,0.523137,MOFA,1_ShareSeq_Skin
4,0.411604,0.615279,0.629103,0.984065,0.566668,Seurat,1_ShareSeq_Skin
